# ROBIN — Quickstart

This notebook walks through loading the 4,900-run archive and reproducing the headline numbers from the paper.


In [ ]:
from robin.data_loader import load_archive
df = load_archive('../data/robin-runs-4900.zip')
print(f'Loaded {len(df)} runs')
df.head()

In [ ]:
canon = df[df['phase'] == 'canonical_eval_16x6x5x4']
summary = canon.assign(closed=(canon['WNS_ns'] >= 0).astype(int)) \
    .groupby('method').agg(n=('WNS_ns','size'),
                          closure_pct=('closed', lambda s: 100*s.mean()),
                          sigma_WNS=('WNS_ns','std')).round(2)
summary

## Conformal coverage


In [ ]:
from robin.conformal import ConformalSignoff
import numpy as np
cal = df[df['phase']=='conformal_calibration_n200']
ho  = df[df['phase']=='conformal_heldout_n400']
pred = cal.groupby('design')['WNS_ns'].mean().to_dict()
for a in (0.05, 0.10, 0.15, 0.20):
    s = ConformalSignoff(alpha=a)
    s.calibrate(cal['design'].map(pred).to_numpy(), cal['WNS_ns'].to_numpy())
    cov = s.empirical_coverage(ho['design'].map(pred).fillna(0).to_numpy(), ho['WNS_ns'].to_numpy())
    print(f'alpha={a:.2f}  coverage={cov:.3f}  q={s.q_alpha:.3f}')